# Setting up maritime costs 

In this notebook, we generate the data-input file network.csv

# Import of libraries

In [2]:
import sys
#import cos
import json
# Stats for joint probability analysis
from scipy.stats import chi2_contingency
from scipy.stats import pearsonr, spearmanr, kendalltau
# Data Manipulation
import pandas as pd
import seaborn as sns
import networkx as nx
from collections import Counter
from  itertools import combinations
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from shapely.geometry import Polygon
import geopandas as gpd
import dask.dataframe as dd
##### Network libraries #######
from geopandas import GeoDataFrame
from pyproj import Geod #### This is to compute the right distance without any bias at the poles.
from itertools import product ### for iterations of pairs
from datetime import datetime
from itertools import combinations # This is for the creation of contigency tables for all pairs of hazards
from shapely.geometry import shape, Point


#Scalers
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

#import osmnx as ox
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points
from shapely.affinity import scale ## This library is needed to rescale polygons to make them bigger. We can then asses how this affects the values of probanilties
import matplotlib.colors as mcolors
from cartopy import crs as ccrs
from tqdm import tqdm  # ✅ Import tqdm



robinson = ccrs.Robinson(central_longitude=0, globe=None)


# Shipping route cost data

We take the data from the supplementary information in "Ports’ criticality in international trade and global supply-chains".



Speed data on different types of vessels is collected in Supplementary Table 3.

In [3]:
speed_params = pd.DataFrame([
    {"mode": "air", "vessel_type": None, "speed_kmh": 750},

    {"mode": "maritime", "vessel_type": "container", "speed_kmh": 30},
    {"mode": "maritime", "vessel_type": "dry_bulk", "speed_kmh": 20},
    {"mode": "maritime", "vessel_type": "general_cargo", "speed_kmh": 22},
    {"mode": "maritime", "vessel_type": "roro", "speed_kmh": 28},
    {"mode": "maritime", "vessel_type": "liquid_bulk", "speed_kmh": 20},

    {"mode": "iww", "vessel_type": None, "speed_kmh": 8},
])

speed_params

,mode,vessel_type,speed_kmh
0,air,None,750
1,maritime,container,30
2,maritime,dry_bulk,20
3,maritime,general_cargo,22
4,maritime,roro,28
5,maritime,liquid_bulk,20
6,iww,None,8


We import cost data from Supplementary Table 7.

In [4]:
cost_params = pd.DataFrame([
    {
        "mode": "air",
        "vessel_type": None,
        "distance_cost_usd_per_tkm": 0.2,
        "time_cost_usd_per_th": 135.0,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "maritime",
        "vessel_type": "container",
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "maritime",
        "vessel_type": "dry_bulk",
        "distance_cost_usd_per_tkm": 0.004,
        "time_cost_usd_per_th": 0.06,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "maritime",
        "vessel_type": "general_cargo",
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "maritime",
        "vessel_type": "roro",
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "maritime",
        "vessel_type": "liquid_bulk",
        "distance_cost_usd_per_tkm": 0.006,
        "time_cost_usd_per_th": 0.11,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "road",
        "vessel_type": None,
        "distance_cost_usd_per_tkm": np.nan,  # country-specific in the paper
        "time_cost_usd_per_th": 4.0,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "rail",
        "vessel_type": None,
        "distance_cost_usd_per_tkm": np.nan,  # country-specific in the paper
        "time_cost_usd_per_th": 1.0,
        "source": "Supplementary Table 7"
    },
    {
        "mode": "iww",
        "vessel_type": None,
        "distance_cost_usd_per_tkm": np.nan,  # country-specific in the paper
        "time_cost_usd_per_th": 0.2,
        "source": "Supplementary Table 7"
    },
])

cost_params

,mode,vessel_type,distance_cost_usd_per_tkm,time_cost_usd_per_th,source
0,air,None,0.200,135.00,Supplementary Table 7
1,maritime,container,0.002,0.03,Supplementary Table 7
2,maritime,dry_bulk,0.004,0.06,Supplementary Table 7
3,maritime,general_cargo,0.002,0.03,Supplementary Table 7
4,maritime,roro,0.002,0.03,Supplementary Table 7
5,maritime,liquid_bulk,0.006,0.11,Supplementary Table 7
6,road,None,NaN,4.00,Supplementary Table 7
7,rail,None,NaN,1.00,Supplementary Table 7
8,iww,None,NaN,0.20,Supplementary Table 7


A combined list can be obtained from

In [5]:
maritime_params = pd.DataFrame([
    {
        "vessel_type": "container",
        "speed_kmh": 30,
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
    },
    {
        "vessel_type": "dry_bulk",
        "speed_kmh": 20,
        "distance_cost_usd_per_tkm": 0.004,
        "time_cost_usd_per_th": 0.06,
    },
    {
        "vessel_type": "general_cargo",
        "speed_kmh": 22,
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
    },
    {
        "vessel_type": "roro",
        "speed_kmh": 28,
        "distance_cost_usd_per_tkm": 0.002,
        "time_cost_usd_per_th": 0.03,
    },
    {
        "vessel_type": "liquid_bulk",
        "speed_kmh": 20,
        "distance_cost_usd_per_tkm": 0.006,
        "time_cost_usd_per_th": 0.11,
    },
])

'\nmaritime_params["time_cost_equiv_usd_per_tkm"] = (\n    maritime_params["time_cost_usd_per_th"] / maritime_params["speed_kmh"]\n)\n\nmaritime_params["total_sailing_cost_usd_per_tkm"] = (\n    maritime_params["distance_cost_usd_per_tkm"]\n    + maritime_params["time_cost_equiv_usd_per_tkm"]\n)\n\nmaritime_params["break_even_speed_kmh"] = (\n    maritime_params["time_cost_usd_per_th"]\n    / maritime_params["distance_cost_usd_per_tkm"]\n)\n\nmaritime_params\n'

# Import of maritime transport edges

In [6]:
path_file = './raw_data/network/edges_maritime_corrected.gpkg'

edges_maritime_gdf = gpd.read_file(path_file)

In [7]:

# Define the geodetic calculator for WGS84 ellipsoid
geod = Geod(ellps="WGS84")

# Compute geodesic distance (in km) for each geometry
edges_maritime_gdf["distance_wgs84"] = edges_maritime_gdf["geometry"].apply(
    lambda geom: geod.geometry_length(geom) / 1000  # Convert meters to km
)


In [8]:
edges_maritime_gdf['dist_difference'] = edges_maritime_gdf['distance'] -  edges_maritime_gdf['distance_wgs84']

# Overlaying spatial maritime transport edge data with transport costs

In [11]:
### We merge the 
edges = edges_maritime_gdf.merge(maritime_params, how="cross")

In [12]:
edges["time_hr"] = edges["distance"] / edges["speed_kmh"]

In [13]:
edges["gcost_usd_per_ton"] = (
    edges["distance_cost_usd_per_tkm"] * edges["distance"]
)

In [14]:
## Saving data to file edges_transport_costs.gpkg
edges.to_file("./raw_data/network/intermediate_data/edges_transport_costs.gpkg", layer="edges", driver="GPKG", index=False)

# Formatting data into useable shape

In [16]:
network_example_df = edges[['from_id', 'to_id','gcost_usd_per_ton','distance_wgs84','time_hr','vessel_type']].copy()

In [17]:
network_example_df['flow_capacity'] = 10**15

In [18]:
network_example_df['id'] = network_example_df['from_id'] + '_' + network_example_df['to_id']

In [19]:
network_example_df = network_example_df.rename(columns = {'distance_wgs84':'length_m'})

In [20]:
network_example_df = network_example_df.drop_duplicates()

# Select a vessel type

In [28]:
network_example_df['vessel_type'].unique()

array(['container', 'dry_bulk', 'general_cargo', 'roro', 'liquid_bulk'],
      dtype=object)

In [29]:
vessel_type = "container" ### The vessel type can be changed here.

In [30]:
network_vessel_df = network_example_df[network_example_df["vessel_type"] == vessel_type].drop(columns="vessel_type").copy()

In [32]:
network_vessel_df.to_csv(
    "./processed_data/network/network.csv",
    index=False,)


# Adapting the configuration file to the vessel type

In [26]:
### Generate a vessel-specific output folder
results_folder = Path("results") / vessel_type
results_folder.mkdir(parents=True, exist_ok=True)

In [27]:
### We want results to be saved in an output file that indicates the vessel type at which we looked. 
### For that we need to adapt the config.maritime.json file to reflect the right results folder path.
import json
from pathlib import Path

# Assumes the notebook runs from the Maritime folder.
config_path = Path("../config.maritime.json")

with config_path.open() as f:
    config = json.load(f)

config["paths"]["results"] = f"./maritime/results/{vessel_type}"

with config_path.open("w") as f:
    json.dump(config, f, indent=2)
    f.write("\n")